# Demo 2 — Concatenate with provenance and deliberate alignment

**Learning objectives**

- Stack same-grain partitions vertically while preserving each row's source.
- Diagnose schema drift through the missing positions created by column-label alignment.
- Concatenate horizontally only when index labels are deliberate row keys, then explain every missing position.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch and rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixture contains invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"
try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None
if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Resolve one pinned visit table

The prepared input has grain one row per visit and a unique `visit_id`. The notebook divides it into two supplied-import partitions only to expose concatenation behavior; it does not introduce datetime or time-series semantics.

In [ ]:
from hashlib import sha256
from pathlib import Path

SOURCE_RELATIVE_PATH = Path("06") / "demo" / "data" / "visits.csv"
EXPECTED_SHA256 = "ccff0b9eaab1b6aae702734628db50b5223b04efc0071e1e9b4b9d6796e0c930"
SUPPLIED_SOURCE_BYTES = (
    b"visit_id,participant_id,visit_number,site_code,status,measure\n"
    b"V001,P01,1,N,complete,12.5\n"
    b"V002,P01,2,N,complete,14.0\n"
    b"V003,P02,1,S,complete,9.5\n"
    b"V004,P03,1,W,complete,11.0\n"
    b"V005,P04,1,N,complete,13.5\n"
    b"V006,P05,1,X,complete,8.0\n"
)


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


DATA_PATH = find_course_file(Path.cwd(), SOURCE_RELATIVE_PATH)
lecture_readme = find_course_file(Path.cwd(), Path("06") / "README.md")
if DATA_PATH is None:
    data_dir = Path.cwd() / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    DATA_PATH = data_dir / "visits.csv"
    DATA_PATH.write_bytes(SUPPLIED_SOURCE_BYTES)
assert sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256

demo_base = Path.cwd() if lecture_readme is None else lecture_readme.parent / "demo"
OUTPUT_DIR = demo_base / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
visits = pd.read_csv(
    DATA_PATH,
    dtype={"visit_id": "string", "participant_id": "string", "site_code": "string", "status": "string"},
)
assert visits["visit_id"].is_unique
print("Input:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)

## Stack same-grain partitions vertically

Both partitions retain the one-row-per-visit grain and the same schema. Add `source_partition` before stacking so provenance remains an ordinary output column. `ignore_index=True` gives the new combined table a fresh positional index; business-key uniqueness is checked separately.

In [ ]:
first_import = visits.iloc[:3].copy().assign(source_partition="first_import")
second_import = visits.iloc[3:].copy().assign(source_partition="second_import")
assert list(first_import.columns) == list(second_import.columns)

combined_visits = pd.concat(
    [first_import, second_import],
    axis="index",
    ignore_index=True,
)
combined_visits["source_partition"] = combined_visits["source_partition"].astype("string")
assert len(combined_visits) == len(first_import) + len(second_import) == 6
assert combined_visits["visit_id"].is_unique
assert combined_visits["source_partition"].value_counts().to_dict() == {"first_import": 3, "second_import": 3}
combined_visits

## Diagnose column-label alignment

Vertical concatenation uses the union of column labels. If one partition loses `measure` and gains `review_note`, missing positions appear. This preview identifies schema drift; it does not decide to fill or delete anything.

In [ ]:
schema_drift_partition = second_import.drop(columns="measure").assign(review_note="late import")
alignment_preview = pd.concat(
    [first_import, schema_drift_partition],
    ignore_index=True,
    sort=False,
)
assert alignment_preview.loc[alignment_preview["source_partition"].eq("second_import"), "measure"].isna().all()
assert alignment_preview.loc[alignment_preview["source_partition"].eq("first_import"), "review_note"].isna().all()
alignment_preview

## Align feature tables horizontally by a documented index key

For `axis='columns'`, pandas aligns by index label rather than row position. Both indexes below are named `visit_id` because they intentionally represent the same visit grain. Labels present on only one side create explainable missing positions.

In [ ]:
measure_by_visit = visits.set_index("visit_id").loc[["V001", "V002", "V003"], ["measure"]]
review_by_visit = pd.DataFrame(
    {"review_score": [7.0, 8.0, 9.0]},
    index=pd.Index(["V002", "V003", "V006"], dtype="string", name="visit_id"),
)
assert measure_by_visit.index.is_unique
assert review_by_visit.index.is_unique

aligned_features = pd.concat([measure_by_visit, review_by_visit], axis="columns")
assert set(aligned_features.index) == {"V001", "V002", "V003", "V006"}
assert aligned_features.isna().sum().to_dict() == {"measure": 1, "review_score": 1}
assert pd.isna(aligned_features.loc["V001", "review_score"])
assert pd.isna(aligned_features.loc["V006", "measure"])
aligned_features

## Replace and verify two generated artifacts

Write the vertical result without its positional index. Preserve the horizontal result's named visit index because that label is the deliberate alignment key, then verify both readbacks.

In [ ]:
COMBINED_PATH = OUTPUT_DIR / "combined_visits.csv"
ALIGNED_PATH = OUTPUT_DIR / "aligned_features.csv"
combined_visits.to_csv(COMBINED_PATH, index=False)
aligned_features.to_csv(ALIGNED_PATH, index=True)

combined_round_trip = pd.read_csv(
    COMBINED_PATH,
    dtype={"visit_id": "string", "participant_id": "string", "site_code": "string", "status": "string", "source_partition": "string"},
)
aligned_round_trip = pd.read_csv(
    ALIGNED_PATH,
    dtype={"visit_id": "string", "measure": "float64", "review_score": "float64"},
).set_index("visit_id")
pd.testing.assert_frame_equal(combined_round_trip, combined_visits)
pd.testing.assert_frame_equal(aligned_round_trip, aligned_features)
print("Demo 2 concat provenance and alignment passed")
print(COMBINED_PATH)
print(ALIGNED_PATH)